In [5]:
import pandas as pd
import numpy as np

# 1. Load the data 
df = pd.read_csv('../data/raw/Fraud_Data.csv')
ip_data = pd.read_csv('../data/raw/IpAddress_to_Country.csv')

# 2. Re-apply essential cleaning (needed so the data is ready for engineering)
df['signup_time'] = pd.to_datetime(df['signup_time'])
df['purchase_time'] = pd.to_datetime(df['purchase_time'])
#Convert all IP addresses to strings first, handling potential non-string types
df['ip_address'] = df['ip_address'].astype(str)
df = df.dropna(subset=['ip_address'])

# 3. Geolocation Integration
def ip_to_int(ip):
    return sum(int(x) * 256**i for i, x in enumerate(reversed(ip.split('.'))))

df['ip_int'] = df['ip_address'].apply(ip_to_int)

# Sort and merge [cite: 108]
df = df.sort_values('ip_int')
# Convert ip_data key to int64 to match df['ip_int']
ip_data['lower_bound_ip_address'] = ip_data['lower_bound_ip_address'].astype('int64')

# Now the merge should work
df = pd.merge_asof(
    df, 
    ip_data, 
    left_on='ip_int', 
    right_on='lower_bound_ip_address', 
    direction='backward'
)

df = pd.merge_asof(df, ip_data, left_on='ip_int', right_on='lower_bound_ip_address', direction='backward')

# 4. Feature Engineering [cite: 110, 111, 112, 115]
df['time_since_signup'] = (df['purchase_time'] - df['signup_time']).dt.total_seconds()
df['hour_of_day'] = df['purchase_time'].dt.hour
df['day_of_week'] = df['purchase_time'].dt.dayofweek
df['device_tx_count'] = df.groupby('device_id')['purchase_time'].transform('count')

# Verify the result
print("Feature Engineering Complete.")
print(df.head())

Feature Engineering Complete.
   user_id         signup_time       purchase_time  purchase_value  \
0   370003 2015-03-03 19:58:39 2015-05-28 21:09:13              33   
1   241175 2015-07-17 11:31:56 2015-08-14 22:21:41              87   
2   109055 2015-06-21 12:29:54 2015-07-17 05:11:55              30   
3   256723 2015-02-12 23:44:59 2015-06-07 02:45:51              14   
4     7601 2015-07-01 01:26:28 2015-09-15 01:47:30              18   

       device_id source browser sex  age        ip_address  ...  \
0  PIBUQMBIELMMG    Ads      IE   M   40   117566.66486748  ...   
1  QXEAKJVUIOMJT    SEO  Chrome   M   33  155399.107924991  ...   
2  HPWKTNSUUMLGU    Ads      IE   M   18   504221.41135231  ...   
3  SSHROXRYPSDZP    Ads      IE   M   34  858558.026950951  ...   
4  NJGTLIKCWSBKR    Ads  Safari   M   29  735354.096794478  ...   

   lower_bound_ip_address_x  upper_bound_ip_address_x      country_x  \
0                  96468992                  96731135        Romania   
1 